In [1]:
import torch
import torch.nn as nn
import numpy as np

# 1. Custom Asymmetric Loss Function for Peak Surges
class AsymmetricPeakLoss(nn.Module):
    def __init__(self, underpredict_penalty=2.5):
        super(AsymmetricPeakLoss, self).__init__()
        self.underpredict_penalty = underpredict_penalty

    def forward(self, y_pred, y_true):
        residual = y_true - y_pred
        # Apply higher weight when residual > 0 (underprediction of grid load)
        loss = torch.where(
            residual > 0,
            self.underpredict_penalty * (residual ** 2),
            residual ** 2
        )
        return torch.mean(loss)

# 2. Target Log Transformation Helper Functions
def transform_target(y):
    return np.log1p(y)  # log(1 + y)

def inverse_transform_target(y_log):
    return np.expm1(y_log)  # exp(y) - 1

# 3. Updated Training Loop Step Example
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = AsymmetricPeakLoss(underpredict_penalty=2.5)

# Example batch step inside training loop
def train_step(model, optimizer, x_batch, y_batch_raw):
    model.train()
    optimizer.zero_grad()
    
    # Apply log scaling to true load target
    y_batch_log = torch.tensor(transform_target(y_batch_raw), dtype=torch.float32).to(device)
    x_batch = x_batch.to(device)
    
    # Forward pass
    predictions_log = model(x_batch)
    loss = criterion(predictions_log, y_batch_log)
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    # Convert predictions back to original kWh for evaluation
    predictions_kwh = inverse_transform_target(predictions_log.detach().cpu().numpy())
    return loss.item(), predictions_kwh